# Orvessa walkthrough

This notebook runs the Orvessa model **one step at a time**, so you can look at the
flows between steps. `run_orvessa.py` does the same thing in one go.

Background reading: `docs/1-the-scenario.md` and `docs/2-how-the-model-works.md`.

In [ ]:
# Make the repository root importable when the notebook is run from notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from trase.models.orvessa.candy import model
from trase.tools.sei_pcs.supply_chain import SupplyChain

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Prepare and load the data

`preparation()` cleans the raw CSVs in `data/` and `load()` reads them in, checking
types and the links declared in `definition.py`.

In [ ]:
sc = SupplyChain("orvessa/candy", 2017)
sc.preparation()
sc.load()

Every customs declaration is now a flow. Linked columns are joined in automatically,
so we can see the exporting asset's level and company next to each shipment:

In [ ]:
COLUMNS = ["asset.name", "asset.level", "asset.company", "country", "vol",
           "branch", "status", "storage.name", "municipality.name", "deforestation"]

sc.get("flows")[COLUMNS]

## 2. Decision tree

Assign each flow to a branch and choose its storage hub.

In [ ]:
model.decision_tree(sc)
sc.get("flows")[COLUMNS]

In [ ]:
sc.flow_report_by_attribute("vol", ["status", "branch"])

The three office exporters (level 5) matched no branch, so they stay `UNKNOWN`.
Everything else is `TO LP` with a hub in `storage.name`.

## 3. Linear programming

Allocate production municipalities to hubs at minimum total distance, then split each
flow across its hub's sources.

In [ ]:
model.linear_programming(sc)
sc.get("flows")[COLUMNS]

Note that there are now more rows than customs declarations: flows whose hub drew on
two municipalities have been split in two. Total volume is unchanged:

In [ ]:
sc.get("flows")["vol"].sum()

## 4. Domestic demand and deforestation

In [ ]:
model.append_domestic_demand(sc)
model.embed_deforestation(sc)
sc.quality_assurance()

sc.get("flows")[COLUMNS]

## 5. Explore the results

Deforestation per tonne by importing country. Remember to check how much of each
country's volume is `UNKNOWN` before comparing them (see `docs/3-reading-the-results.md`).

In [ ]:
df = sc.get("flows")
summary = df.groupby("country").agg(
    volume_t=("vol", "sum"),
    deforestation_ha=("deforestation", "sum"),
    untraced_t=("vol", lambda v: v[df.loc[v.index, "status"] == "UNKNOWN"].sum()),
)
summary["ha_per_t"] = summary["deforestation_ha"] / summary["volume_t"]
summary.sort_values("deforestation_ha", ascending=False).round(1)

A Sankey diagram of the supply chain, from municipality of production through exporter
to destination (requires `plotly`):

In [ ]:
sc.sankey(
    "vol",
    ["municipality.name", "asset.company", "country"],
    title_text="Orvessa candy flows, 2017 (tonnes)",
)

In [ ]:
# write the results file (trase/models/orvessa/candy/2017/results/results.csv)
sc.export_results()
sc.results_path

## Try it yourself

- Change a number in `trase/models/orvessa/candy/data/` and re-run the notebook.
- Work through `docs/5-exercises.md`.